<a href="https://colab.research.google.com/github/Pradeep1694/Capstone-Project/blob/main/Capstone_Project_Part_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [368]:
import numpy as np
import pandas as pd
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.impute import SimpleImputer

df = pd.read_csv("cleaned_data.csv")

X = df.drop(columns=["discounted_price"])

# Encode Categorical Columns
drop_cols = [
    "product_id",
    "product_name",
    "about_product",
    "user_id",
    "user_name",
    "review_id",
    "review_title",
    "review_content",
    "img_link",
    "product_link"
]

X = X.drop(columns=drop_cols)

X = pd.get_dummies(X,columns=["category"], drop_first=True)
y_reg = df["discounted_price"]
y_clf = (y_reg >= y_reg.median()).astype(int)


x_train, x_test, y_clf_train, y_clf_test = train_test_split(X, y_clf, test_size=0.2, random_state=42)


scaler = StandardScaler()
x_train_scaler = scaler.fit_transform(x_train)
x_test_scaler = scaler.transform(x_test)


dt = DecisionTreeClassifier(random_state=42)
dt.fit(x_train_scaler, y_clf_train)


train_pred = dt.predict(x_train_scaler)
test_pred = dt.predict(x_test_scaler)


print(f"Training Accuracy: {accuracy_score(y_clf_train, train_pred):.4f}")
print(f"Test Accuracy: {accuracy_score(y_clf_test, test_pred):.4f}")

Training Accuracy: 1.0000
Test Accuracy: 0.9863


In [369]:
# Controlled Decision Tree
dt_control = DecisionTreeClassifier(max_depth=5, min_samples_split=20, random_state=42)

dt_control.fit(x_train_scaler,y_clf_train)

train_pred = dt_control.predict(x_train_scaler)
test_pred = dt_control.predict(x_test_scaler)

print("Training Accuracy: ", accuracy_score(y_clf_train, train_pred))
print("Test Accuracy: ", accuracy_score(y_clf_test, test_pred))

Training Accuracy:  0.985494880546075
Test Accuracy:  0.9795221843003413


In [370]:
# Gini vs Entropy comparison
gini = DecisionTreeClassifier(criterion="gini", max_depth=5, random_state=42)

entropy = DecisionTreeClassifier(criterion="entropy", max_depth=5, random_state=42)

gini.fit(x_train_scaler, y_clf_train)
entropy.fit(x_train_scaler, y_clf_train)
gini_acc = accuracy_score(y_clf_test, gini.predict(x_test_scaler))

entropy_acc =accuracy_score(y_clf_test, entropy.predict(x_test_scaler))

print("Gini Accuracy:", gini_acc)
print("Entropy Accuracy:", entropy_acc)

Gini Accuracy: 0.9863481228668942
Entropy Accuracy: 0.9897610921501706


In [371]:
# Random Forest
from sklearn.ensemble import RandomForestClassifier

random_forest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)

random_forest.fit(x_train_scaler, y_clf_train)

random_pred = random_forest.predict(x_test_scaler)
random_prob = random_forest.predict_proba(x_test_scaler)
print("Random Forest Accuracy:", accuracy_score(y_clf_test, random_pred))

print("ROC AUC:, ", roc_auc_score(y_clf_test, random_prob[:, 1]))

Random Forest Accuracy: 0.9419795221843004
ROC AUC:,  0.9951994780014914


In [372]:
important = pd.DataFrame({"Feature": x_train.columns, "Importance": random_forest.feature_importances_})

important = important.sort_values(by="Importance", ascending=False)

print(important)
print("\n Top 5 Features:", important.head())

                                               Feature  Importance
0                                         actual_price    0.523841
1                                  discount_percentage    0.107801
13   category_Computers&Accessories|Accessories&Per...    0.057738
3                                         rating_count    0.040869
122  category_Electronics|WearableTechnology|SmartW...    0.033718
..                                                 ...         ...
196  category_HomeImprovement|Electrical|Adapters&M...    0.000000
197  category_HomeImprovement|Electrical|CordManage...    0.000000
203  category_OfficeProducts|OfficePaperProducts|Pa...    0.000000
202  category_OfficeProducts|OfficePaperProducts|Pa...    0.000000
211  category_OfficeProducts|OfficePaperProducts|Pa...    0.000000

[214 rows x 2 columns]

 Top 5 Features:                                                Feature  Importance
0                                         actual_price    0.523841
1                   

In [373]:
# Gradient Boosting
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(max_iter=100, learning_rate=0.1, max_depth=3, random_state=42)
gb.fit(x_train_scaler, y_clf_train)

gb_pred = gb.predict(x_test_scaler)
gb_prob = gb.predict_proba(x_test_scaler)

print("Gradient Boosting Accuracy:", accuracy_score(y_clf_test, gb_pred))
print("ROC AUC:", roc_auc_score(y_clf_test, gb_prob[:, 1]))

Gradient Boosting Accuracy: 0.9931740614334471
ROC AUC: 0.9997203579418344


In [374]:
# Feature ablation study
least5 = important.tail(5)
least5_features = least5["Feature"].tolist()

x_train_reduced = x_train.drop(columns=least5_features)
x_test_reduced = x_test.drop(columns=least5_features)

scaler = StandardScaler()
x_train_red = scaler.fit_transform(x_train_reduced)
x_test_red = scaler.transform(x_test_reduced)

random_forest2 = RandomForestClassifier(random_state=42)
random_forest2.fit(x_train_red, y_clf_train)

auc_full = roc_auc_score(y_clf_test, random_forest.predict_proba(x_test_scaler)[:, 1])
auc_reduced = roc_auc_score(y_clf_test, random_forest2.predict_proba(x_test_red)[:, 1])

print("AUC Full:", auc_full)
print("AUC Reduced:", auc_reduced)

AUC Full: 0.9951994780014914
AUC Reduced: 0.9994873228933632


In [375]:
# Cross-validated comparison:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


# Cross-validated comparison
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    "Logistic": LogisticRegression(max_iter=1000),
    "Decision Tree": dt_control,
    "Random Forest": random_forest,
    "Gradient Boosting": gb
}
for name, model in models.items():
    pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), model)
    scores = cross_val_score(pipeline, x_train, y_clf_train, cv=cv, scoring="roc_auc")
    print(name)
    print(f"Mean AUC: {scores.mean():.4f}")
    print(f"Standard Deviation: {scores.std():.4f}")
    print()

Logistic
Mean AUC: 0.9398
Standard Deviation: 0.0096

Decision Tree
Mean AUC: 0.9691
Standard Deviation: 0.0165

Random Forest
Mean AUC: 0.9905
Standard Deviation: 0.0037

Gradient Boosting
Mean AUC: 0.9988
Standard Deviation: 0.0007



In [376]:
# Hyperparameter tuning with GridSearchCV
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Hyperparameter tuning with GridSearchCV

pipeline = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), RandomForestClassifier(random_state=42))

param_grid = {
    'randomforestclassifier__n_estimators': [50, 100, 200],
    'randomforestclassifier__max_depth': [5, 10, None],
    'randomforestclassifier__min_samples_leaf': [1, 5]
}


grid = GridSearchCV(pipeline, param_grid, cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42), scoring='roc_auc', n_jobs=-1)
grid.fit(x_train, y_clf_train)
print(grid.best_params_)
print(grid.best_score_)

{'randomforestclassifier__max_depth': None, 'randomforestclassifier__min_samples_leaf': 1, 'randomforestclassifier__n_estimators': 200}
0.9965531603945926


In [377]:
# Manual learning curve
import pandas as pd
from sklearn.metrics import roc_auc_score

fractions = [0.2, 0.4, 0.6, 0.8, 1.0]

results=[]

for f in fractions:
  n = int(f * len (x_train))

  x_sub = x_train.iloc[:n]
  y_sub = y_clf_train.iloc[:n]

  grid.best_estimator_.fit(x_sub, y_sub)
  train_auc = roc_auc_score(y_sub, grid.best_estimator_.predict_proba(x_sub)[:,1])

  test_auc = roc_auc_score(y_clf_test,grid.best_estimator_.predict_proba(x_test)[:,1])

  results.append([f, train_auc, test_auc])

learning_curve =pd.DataFrame(results, columns=["Fraction", "Train AUC", "Test AUC"])
print(learning_curve)

   Fraction  Train AUC  Test AUC
0       0.2        1.0  0.994151
1       0.4        1.0  0.997413
2       0.6        1.0  0.997763
3       0.8        1.0  0.998742
4       1.0        1.0  0.999511


In [378]:
# Serialize the best model:
import joblib

joblib.dump(grid.best_estimator_, "best_model.pk1")

model = joblib.load("best_model.pk1")

pred = model.predict(x_test.iloc[:2])
print(pred)

[1 0]
